# 00 · Instrument calibration

**Grand Challenge Labs · Coupling-Phase Spectroscopy**

Calibrate the CPS instrument before asking it to support a model-level claim. This notebook checks the software path, core mathematical contracts, and the synthetic reference experiment.


## Release contract

| Contract | Declared value |
|---|---|
| **Scientific question** | Does this runtime reproduce the CPS instrument contracts and synthetic reference behavior? |
| **Default path** | Run a focused unit-test slice, then execute the controlled synthetic quadratic experiment. |
| **Evidence boundary** | A pass establishes implementation readiness. It does not establish an empirical claim about Pythia training. |
| **Primary outputs** | Test results, synthetic evidence artifacts, and `/content/cps-export.zip`. |


## Interpretation checklist

- [ ] Confirm that CPS imports from the checked-out repository source tree.
- [ ] Treat any failed contract test as an instrument failure; do not interpret later model notebooks until it is repaired.
- [ ] Verify that the synthetic experiment emits a complete evidence packet before export.


## Learning objectives

By the end of the run, you should be able to distinguish:

1. a phase-family construction from an arbitrary perturbation;
2. eigenvalue continuation from independent eigenvalue sorting;
3. asymptotic spectral radius from finite-horizon transient gain;
4. a software smoke test from empirical evidence about Pythia training.

**Evidence boundary.** Passing this notebook validates the implementation and synthetic fixtures. It does not validate any claim about a language model or optimizer trajectory.

In [ ]:
import os, pathlib, subprocess, sys, time
from IPython.display import Markdown, display

REPO_URL = os.environ.get("CPS_REPO_URL", "https://github.com/fyremael/CPS.git")
GIT_REF = os.environ.get("CPS_GIT_REF", "main")
repo = pathlib.Path("/content/CPS")

print("[BOOT] Preparing the CPS repository", flush=True)
print(f"[BOOT] source={REPO_URL}", flush=True)
print(f"[BOOT] ref={GIT_REF}", flush=True)
if not repo.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", GIT_REF, REPO_URL, str(repo)], check=True)
else:
    subprocess.run(["git", "-C", str(repo), "fetch", "origin", GIT_REF], check=True)
    subprocess.run(["git", "-C", str(repo), "checkout", GIT_REF], check=True)
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only", "origin", GIT_REF], check=True)
os.chdir(repo)
print("[BOOT] Installing CPS with Pythia and notebook dependencies", flush=True)
subprocess.run([
    sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
    "-e", ".[pythia,notebooks]"
], check=True)
print(f"[BOOT] Ready: {repo}", flush=True)

# Editable installs write a .pth file, but the running Colab kernel does not
# automatically reprocess newly-created .pth files. Put the source tree on
# sys.path explicitly so the very next cell can import CPS without a restart.
import importlib
src_dir = repo / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))
importlib.invalidate_caches()
import cps
print(f"[BOOT] CPS import verified from {cps.__file__}", flush=True)


In [ ]:
from cps.notebook import apply_release_theme, stage_banner
stage_banner(
    "0",
    "Record the runtime contract",
    objective="Expose the software and accelerator environment before scientific execution.",
    deliverable="A visible runtime inventory for reproducibility.",
)
from cps.notebook import show_environment
apply_release_theme()
runtime = show_environment()

## Stage 1 — test the structural contracts

The selected tests exercise magnitude-preserving perturbations, eigenvalue tracking, projected operators, and the new JVP fallback policy. Verbose test names are shown so failures are locally interpretable.

In [ ]:
from cps.notebook import stage_banner
stage_banner('1', 'test the structural contracts', objective='The selected tests exercise magnitude-preserving perturbations, eigenvalue tracking, projected operators, and the new JVP fallback policy. Verbose test names are shown so failures are locally interpretable.', deliverable="A verified stage result recorded in the evidence packet.")

import subprocess, sys
command = [
    sys.executable, "-m", "pytest", "-vv",
    "tests/test_perturbations.py",
    "tests/test_spectra.py",
    "tests/pythia/test_reduced_operator.py",
    "tests/pythia/test_jvp.py",
]
print("[TEST]", " ".join(command), flush=True)
subprocess.run(command, check=True)

## Stage 2 — generate a synthetic optimizer-like example

The synthetic experiment supplies a controlled matrix for which spectral motion and transient amplification can be inspected without checkpoint, tokenizer, or fused-kernel confounders.

In [ ]:
from cps.notebook import stage_banner
stage_banner('2', 'generate a synthetic optimizer-like example', objective='The synthetic experiment supplies a controlled matrix for which spectral motion and transient amplification can be inspected without checkpoint, tokenizer, or fused-kernel confounders.', deliverable="A verified stage result recorded in the evidence packet.")

import subprocess, sys
print("[SYNTHETIC] Generating the quadratic-system evidence packet", flush=True)
subprocess.run([sys.executable, "experiments/synthetic_quadratics.py"], check=True)
print("[SYNTHETIC] Complete", flush=True)

## Final stage — export the evidence packet

Every release notebook ends with the same preservation step. The archive contains the evidence produced in this runtime and is suitable for Colab CLI retrieval or manual download.


In [ ]:
from cps.notebook import export_artifacts, stage_banner

stage_banner(
    "EXPORT",
    "Package the evidence",
    objective="Collect the run artifacts into one portable archive.",
    deliverable="/content/cps-export.zip",
)
archive = export_artifacts()
print(f"[EXPORT] archive={archive}", flush=True)
